In [1]:
import pandas as pd, numpy as np, json
from pathlib import Path

ROOT = Path(r"c:\Ground water")
DATA = ROOT / "data" / "processed"

df = pd.read_csv(DATA / "dataset_v2_3.csv", dtype={"SITE_ID": str}, low_memory=False)
with open(DATA / "feature_groups_v2_3.json") as f:
    fg = json.load(f)

TARGET = fg["target"]
tr = df[df.SPLIT == "train"]; va = df[df.SPLIT == "val"]; te = df[df.SPLIT == "test"]
print(df.shape, len(tr), len(va), len(te))

(92452, 77) 50363 1859 5178


In [1]:
import os, sys
import numpy as np
os.chdir(r"C:\Ground water")
sys.path.append(r"C:\Ground water\scripts")
import model_utils as mu

d = mu.load_data()
train, val, test = mu.split(d)
lag = "water_level_lag_1_obs_m"
target = "WLS_WTR_LEVEL"

for name, part in [("val", val), ("test", test)]:
    print("=====", name, "| rows:", len(part), "| lag NaN %:", round(part[lag].isna().mean() * 100, 1))
    print("persistence:", mu.persistence_scores(part))
    mean_pred = np.full(len(part), train[target].mean())
    print("train-mean baseline:", mu.scores(part[target], mean_pred))
    

===== val | rows: 1859 | lag NaN %: 0.0
persistence: {'n': 1859, 'R2': 0.9737, 'MAE': 2.07, 'MedAE': 1.02, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean baseline: {'n': 1859, 'R2': -0.002, 'MAE': 17.545, 'MedAE': 15.203}
===== test | rows: 5178 | lag NaN %: 0.0
persistence: {'n': 5178, 'R2': 0.9403, 'MAE': 2.648, 'MedAE': 1.155, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean baseline: {'n': 5178, 'R2': -0.0003, 'MAE': 17.311, 'MedAE': 15.143}


In [2]:
import os, sys
import numpy as np
os.chdir(r"C:\Ground water")
sys.path.append(r"C:\Ground water\scripts")
import model_utils as mu

d = mu.load_data()
train, val, test = mu.split(d)
target = "WLS_WTR_LEVEL"
train_mean = train[target].mean()
print("train mean level:", round(train_mean, 3))

for name, part in [("val", val), ("test", test)]:
    print("=====", name, "| rows:", len(part))
    print("persistence  :", mu.persistence_scores(part))
    print("train-mean   :", mu.scores(part[target], np.full(len(part), train_mean)))

train mean level: 21.243
===== val | rows: 1859
persistence  : {'n': 1859, 'R2': 0.9737, 'MAE': 2.07, 'MedAE': 1.02, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean   : {'n': 1859, 'R2': -0.002, 'MAE': 17.545, 'MedAE': 15.203}
===== test | rows: 5178
persistence  : {'n': 5178, 'R2': 0.9403, 'MAE': 2.648, 'MedAE': 1.155, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
train-mean   : {'n': 5178, 'R2': -0.0003, 'MAE': 17.311, 'MedAE': 15.143}


In [3]:
cols = [c for c in d.columns if any(k in c.lower() for k in ["date", "month", "round", "season", "period"])]
print(cols)
print(test[cols].head())

['DATE', 'MONTH', 'ROUND', 'month_sin', 'month_cos', 'ASSESSMENT_DATE', 'level_same_round_prev_year_m']
           DATE  MONTH ROUND  month_sin  month_cos ASSESSMENT_DATE  \
122  2022-01-30      1   Jan   0.500000   0.866025      2023-03-31   
123  2022-05-30      5   May   0.500000  -0.866025      2023-03-31   
124  2022-08-30      8   Aug  -0.866025  -0.500000      2023-03-31   
125  2022-11-10     11   Nov  -0.500000   0.866025      2023-03-31   
126  2023-01-10      1   Jan   0.500000   0.866025      2023-03-31   

     level_same_round_prev_year_m  
122                          4.98  
123                           NaN  
124                          4.73  
125                          5.23  
126                          6.63  


In [4]:
import json, sklearn
import numpy as np

lag = "water_level_lag_1_obs_m"
target = "WLS_WTR_LEVEL"

def by_round(part):
    out = {}
    for r in ["Jan", "May", "Aug", "Nov"]:
        p = part[part["ROUND"] == r]
        out[r] = mu.persistence_scores(p) if len(p) > 0 else {"n": 0}
    return out

res = {
    "dataset_version": "dataset_v2_3.csv",
    "rows_total": int(len(d)),
    "split_rows": {"train": int(len(train)), "val": int(len(val)), "test": int(len(test))},
    "split_years": {"train": "2005-2020", "val": "2021", "test": "2022-2023"},
    "random_state": 42,
    "sklearn_version": sklearn.__version__,
    "train_mean_level": round(float(train[target].mean()), 3),
    "persistence": {"val": mu.persistence_scores(val), "test": mu.persistence_scores(test)},
    "train_mean_baseline": {
        "val": mu.scores(val[target], np.full(len(val), train[target].mean())),
        "test": mu.scores(test[target], np.full(len(test), train[target].mean()))},
    "persistence_by_round": {"val": by_round(val), "test": by_round(test)},
}

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
with open(r"C:\Ground water\docs\results\metrics_baseline.json", "w") as f:
    json.dump(res, f, indent=2)

for split in ["val", "test"]:
    print("=====", split, "round-wise persistence")
    for r, s in res["persistence_by_round"][split].items():
        print(r, s)

===== val round-wise persistence
Jan {'n': 617, 'R2': 0.9748, 'MAE': 1.79, 'MedAE': 0.79, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 83, 'R2': 0.9751, 'MAE': 2.922, 'MedAE': 2.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 560, 'R2': 0.9654, 'MAE': 2.452, 'MedAE': 1.3, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 598, 'R2': 0.9774, 'MAE': 1.887, 'MedAE': 0.805, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
===== test round-wise persistence
Jan {'n': 1296, 'R2': 0.9526, 'MAE': 2.182, 'MedAE': 0.8, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 1295, 'R2': 0.9376, 'MAE': 2.762, 'MedAE': 1.35, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 1284, 'R2': 0.9203, 'MAE': 3.386, 'MedAE': 1.82, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 1279, 'R2': 0.9597, 'MAE': 2.176, 'MedAE': 1.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}


In [6]:
import json, sklearn
import numpy as np

target = "WLS_WTR_LEVEL"

def by_round(part):
    out = {}
    for r in ["Jan", "May", "Aug", "Nov"]:
        p = part[part["ROUND"] == r]
        out[r] = mu.persistence_scores(p) if len(p) > 0 else {"n": 0}
    return out

res = {
    "dataset_version": "dataset_v2_3.csv",
    "rows_total": int(len(d)),
    "split_rows": {"train": int(len(train)), "val": int(len(val)), "test": int(len(test))},
    "split_years": {"train": "2005-2020", "val": "2021", "test": "2022-2023"},
    "random_state": 42,
    "sklearn_version": sklearn.__version__,
    "train_mean_level": round(float(train[target].mean()), 3),
    "persistence": {"val": mu.persistence_scores(val), "test": mu.persistence_scores(test)},
    "train_mean_baseline": {
        "val": mu.scores(val[target], np.full(len(val), train[target].mean())),
        "test": mu.scores(test[target], np.full(len(test), train[target].mean()))},
    "persistence_by_round": {"val": by_round(val), "test": by_round(test)},
}

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
with open(r"C:\Ground water\docs\results\metrics_baseline.json", "w") as f:
    json.dump(res, f, indent=2)

for split in ["val", "test"]:
    print("=====", split, "round-wise persistence")
    for r, s in res["persistence_by_round"][split].items():
        print(r, s)

===== val round-wise persistence
Jan {'n': 617, 'R2': 0.9748, 'MAE': 1.79, 'MedAE': 0.79, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 83, 'R2': 0.9751, 'MAE': 2.922, 'MedAE': 2.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 560, 'R2': 0.9654, 'MAE': 2.452, 'MedAE': 1.3, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 598, 'R2': 0.9774, 'MAE': 1.887, 'MedAE': 0.805, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
===== test round-wise persistence
Jan {'n': 1296, 'R2': 0.9526, 'MAE': 2.182, 'MedAE': 0.8, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
May {'n': 1295, 'R2': 0.9376, 'MAE': 2.762, 'MedAE': 1.35, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Aug {'n': 1284, 'R2': 0.9203, 'MAE': 3.386, 'MedAE': 1.82, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}
Nov {'n': 1279, 'R2': 0.9597, 'MAE': 2.176, 'MedAE': 1.0, 'rows_dropped_lag_nan': 0, 'dropped_pct': 0.0}


In [7]:
g = mu.load_groups()
for k, v in g.items():
    print(k, "->", len(v), v[:6])

target -> 13 WLS_WT
baseline -> 6 ['DISTRICT_NAME', 'LAT', 'LON', 'YEAR', 'month_sin', 'month_cos']
rainfall -> 11 ['rainfall_mm', 'rainfall_lag_1m_mm', 'rainfall_lag_3m_mm', 'rainfall_lag_12m_mm', 'rainfall_normal_train_mm', 'rainfall_anomaly_mm']
elevation -> 1 ['elevation_m']
well_aquifer -> 2 ['WELL_TYPE', 'AQUIFER_TYPE']
lag -> 9 ['water_level_lag_1_obs_m', 'water_level_lag_2_obs_m', 'water_level_prev3_obs_mean_m', 'water_level_prev5_obs_mean_m', 'days_since_previous_observation', 'level_trend_lag1_lag2_m']
neighbor -> 3 ['neighbor_prev_level_mean_5_m', 'nearest_neighbor_prev_distance_km', 'neighbor_prev_level_count_5']
stage_2023_snapshot -> 2 ['STAGE_EXTRACTION_PCT', 'CATEGORY_CODE']
never_use_as_features -> 3 ['WELL_DEPTH', 'LEVEL_DIFF', 'OUTLIER']
ids_and_bookkeeping -> 22 ['SITE_ID', 'STATION_NAME', 'DATE', 'MONTH', 'STATE_NAME', 'SOURCE_DS']
stage_district_2023_snapshot -> 3 ['district_stage_mean_pct', 'district_overexploited_share', 'district_n_blocks']
unverified_do_not_us

In [8]:
from sklearn.ensemble import RandomForestRegressor

target = "WLS_WTR_LEVEL"
print("variant_B_groups:", g["variant_B_groups"])
print("variant_A_groups:", g["variant_A_groups"])

cols_with_year = mu.cols_for(g, ["baseline"])
cols_no_year = [c for c in cols_with_year if c != "YEAR"]

for label, cols in [("baseline, YEAR ke saath", cols_with_year), ("baseline, YEAR ke bina", cols_no_year)]:
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    pipe = mu.make_pipeline(model, cols, train)
    pipe.fit(train[cols], train[target])
    print(label, "| features:", len(cols), "|", mu.scores(val[target], pipe.predict(val[cols])))

variant_B_groups: ['baseline', 'rainfall', 'elevation', 'well_aquifer', 'neighbor', 'stage_2023_snapshot', 'stage_district_2023_snapshot']
variant_A_groups: ['baseline', 'rainfall', 'elevation', 'well_aquifer', 'neighbor', 'stage_2023_snapshot', 'stage_district_2023_snapshot', 'lag']
baseline, YEAR ke saath | features: 6 | {'n': 1859, 'R2': 0.9759, 'MAE': 2.15, 'MedAE': 1.205}
baseline, YEAR ke bina | features: 5 | {'n': 1859, 'R2': 0.9448, 'MAE': 3.411, 'MedAE': 2.042}


In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)

cols_with_year = mu.cols_for(g, ["baseline"])
cols_no_year = [c for c in cols_with_year if c != "YEAR"]

for label, cols in [("baseline, YEAR ke saath", cols_with_year), ("baseline, YEAR ke bina", cols_no_year)]:
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    pipe = mu.make_pipeline(model, cols, train)
    oof = cross_val_predict(pipe, train[cols], train[target], cv=cv, groups=groups)
    print("NAYA STATION |", label, "|", mu.scores(train[target], oof))

NAYA STATION | baseline, YEAR ke saath | {'n': 50363, 'R2': 0.5442, 'MAE': 8.602, 'MedAE': 4.67}
NAYA STATION | baseline, YEAR ke bina | {'n': 50363, 'R2': 0.5294, 'MAE': 8.88, 'MedAE': 5.0}


In [10]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)
forbidden = set(g["never_use_as_features"]) | {target}

rain_core_names = ["rainfall_mm", "rainfall_lag_1m_mm", "rainfall_lag_3m_mm", "rainfall_lag_12m_mm", "rainfall_normal_train_mm", "rainfall_anomaly_mm"]
rain_core = [c for c in g["rainfall"] if c in rain_core_names]
rain_new5 = [c for c in g["rainfall"] if c not in rain_core_names]
print("rain_core:", rain_core)
print("rain_new5:", rain_new5)

base = list(g["baseline"])
steps = [
    ("baseline", base),
    ("+rain_core", rain_core),
    ("+rain_new5", rain_new5),
    ("+elevation", list(g["elevation"])),
    ("+well_aquifer", list(g["well_aquifer"])),
    ("+neighbor", list(g["neighbor"])),
    ("+stage_district", list(g["stage_district_2023_snapshot"])),
]
extras = [
    ("+stage_block (alag row)", list(g["stage_2023_snapshot"])),
    ("+climate_extra (alag row)", list(g["climate_extra"])),
    ("+evap_soil_extra (alag row)", list(g["evap_soil_extra"])),
]

def run(label, cols):
    assert not (set(cols) & forbidden), "forbidden column mila!"
    model = RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1)
    pipe = mu.make_pipeline(model, cols, train)
    oof = cross_val_predict(pipe, train[cols], train[target], cv=cv, groups=groups)
    s = mu.scores(train[target], oof)
    print(label, "| features:", len(cols), "|", s, flush=True)
    return {"step": label, "n_features": len(cols), "R2": s["R2"], "MAE": s["MAE"], "MedAE": s["MedAE"]}

rows, cur = [], []
for label, add in steps:
    cur = cur + add
    rows.append(run(label, cur))
full = list(cur)
for label, add in extras:
    rows.append(run(label, full + add))

os.makedirs(r"C:\Ground water\docs\results", exist_ok=True)
pd.DataFrame(rows).to_csv(r"C:\Ground water\docs\results\ablation_table.csv", index=False)
print("saved ablation_table.csv")

rain_core: ['rainfall_mm', 'rainfall_lag_1m_mm', 'rainfall_lag_3m_mm', 'rainfall_lag_12m_mm', 'rainfall_normal_train_mm', 'rainfall_anomaly_mm']
rain_new5: ['last_monsoon_total_mm', 'last_monsoon_anomaly_mm', 'rain_cum_6m_mm', 'rain_cum_12m_mm', 'rain_cum_12m_anomaly_mm']
baseline | features: 6 | {'n': 50363, 'R2': 0.5409, 'MAE': 8.605, 'MedAE': 4.646}
+rain_core | features: 12 | {'n': 50363, 'R2': 0.5404, 'MAE': 8.604, 'MedAE': 4.653}
+rain_new5 | features: 17 | {'n': 50363, 'R2': 0.5463, 'MAE': 8.529, 'MedAE': 4.594}
+elevation | features: 18 | {'n': 50363, 'R2': 0.6152, 'MAE': 8.229, 'MedAE': 4.842}
+well_aquifer | features: 20 | {'n': 50363, 'R2': 0.622, 'MAE': 8.121, 'MedAE': 4.856}
+neighbor | features: 23 | {'n': 50363, 'R2': 0.6661, 'MAE': 7.68, 'MedAE': 4.519}
+stage_district | features: 26 | {'n': 50363, 'R2': 0.67, 'MAE': 7.646, 'MedAE': 4.509}
+stage_block (alag row) | features: 28 | {'n': 50363, 'R2': 0.6624, 'MAE': 7.753, 'MedAE': 4.578}
+climate_extra (alag row) | featur

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict

target = "WLS_WTR_LEVEL"
groups = train["SITE_ID"].astype(str)
cv = GroupKFold(n_splits=5)
forbidden = set(g["never_use_as_features"]) | {target}

colsA = mu.cols_for(g, g["variant_A_groups"])
assert not (set(colsA) & forbidden), "forbidden column mila!"
print("Variant A features:", len(colsA))

model = RandomForestRegressor(n_estimators=60, random_state=42, n_jobs=-1)
pipe = mu.make_pipeline(model, colsA, train)
oof = cross_val_predict(pipe, train[colsA], train[target], cv=cv, groups=groups)

ok = train["water_level_lag_1_obs_m"].notna().values
print("train rows lag NaN:", int((~ok).sum()))
print("Variant A (naya station, sab rows)   :", mu.scores(train[target], oof))
print("Variant A (naya station, lag wali rows):", mu.scores(train.loc[ok, target], oof[ok]))
print("persistence (train, same rows)       :", mu.persistence_scores(train))

Variant A features: 37
train rows lag NaN: 620
Variant A (naya station, sab rows)   : {'n': 50363, 'R2': 0.9599, 'MAE': 2.197, 'MedAE': 1.176}
Variant A (naya station, lag wali rows): {'n': 49743, 'R2': 0.9668, 'MAE': 2.091, 'MedAE': 1.155}
persistence (train, same rows)       : {'n': 49743, 'R2': 0.9538, 'MAE': 2.327, 'MedAE': 1.04, 'rows_dropped_lag_nan': 620, 'dropped_pct': 1.23}
